In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

df = pd.read_csv('../heart.csv')

In [ ]:
df

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

target_col = "output" if "output" in df.columns else df.columns[-1]
feature_cols = [c for c in df.columns if c != target_col]

X_all = df[feature_cols].values
y_all = df[target_col].values

# Wyodrębniamy test finalny (20%) i resztę do dalszych podziałów
X_traindev, X_test_final, y_traindev, y_test_final = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)
# Wyodrębniamy zestaw dev służący do wyboru najlepszego modelu (25% z pozostałych 80%)
X_train_sel, X_dev, y_train_sel, y_dev = train_test_split(
    X_traindev, y_traindev, test_size=0.25, random_state=42  # 0.25 * 0.8 = 0.2 overall
)

results = []

# Sprawdzamy wszystkie pary kolumn, bez powtórzeń (i,j) i (j,i)
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        pair = [feature_cols[i], feature_cols[j]]
        idx_i = feature_cols.index(pair[0])
        idx_j = feature_cols.index(pair[1])

        # Trenujemy model KNN na wybranych dwóch kolumnach
        model_pair = KNeighborsClassifier(n_neighbors=2)
        model_pair.fit(X_train_sel[:, [idx_i, idx_j]], y_train_sel)
        y_pred_dev = model_pair.predict(X_dev[:, [idx_i, idx_j]])

        # Oceniamy dokładność na zbiorze dev
        acc = accuracy_score(y_dev, y_pred_dev)
        results.append((pair, acc))

results = sorted(results, key=lambda x: x[1], reverse=True)
best_pair = results[0][0]

print("Najlepsza para kolumn:", best_pair)
print("Dokładność na zbiorze dev:", results[0][1])

print("\nTop 10 par:")
for pair, acc in results[:10]:
    print(f"{pair}: {acc:.4f}")

# Ostateczne trenowanie na pełnym zbiorze traindev
idx_i = feature_cols.index(best_pair[0])
idx_j = feature_cols.index(best_pair[1])
final_model = KNeighborsClassifier(n_neighbors=1)
final_model.fit(X_traindev[:, [idx_i, idx_j]], y_traindev)  # retrain on full traindev

# Ocena na zbiorze testowym
y_pred_test = final_model.predict(X_test_final[:, [idx_i, idx_j]])
print("\nDokładność na zbiorze testowym (held-out):", accuracy_score(y_test_final, y_pred_test))